In [21]:
import pandas as pd
import numpy as np
df=pd.read_csv("C:/Users/lenovo/OneDrive/Desktop/Heart/datasets_4123_6408_framingham.csv")
df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [22]:
df.drop('education',axis=1,inplace=True)

In [23]:
bin_col=['cigsPerDay','BPMeds','totChol','BMI','heartRate','glucose']
for col in bin_col:
    median_val=df[col].median()
    df[col].fillna(median_val,inplace=True)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_14776\195027828.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val,inplace=True)


In [24]:
df['TenYearCHD'].value_counts()

TenYearCHD
0    3596
1     644
Name: count, dtype: int64

In [25]:
from sklearn.utils import resample
df_majority=df[df['TenYearCHD']==0]
df_minority=df[df['TenYearCHD']==1]
df_minority_upsampled=resample(df_minority,replace=True,
                               n_samples=len(df_majority),
                               random_state=42)

In [26]:
df_balanced=pd.concat([df_majority,df_minority_upsampled])

In [27]:
df_balanced['TenYearCHD'].value_counts()


TenYearCHD
0    3596
1    3596
Name: count, dtype: int64

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
x=df_balanced.drop(columns=['TenYearCHD'])
y=df_balanced['TenYearCHD']

In [29]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.fit_transform(x_test)

In [30]:
from sklearn.ensemble import RandomForestClassifier
rf_classifier=RandomForestClassifier()
rf_classifier.fit(x_train_scaled,y_train)
y_pred_rf=rf_classifier.predict(x_test_scaled)

In [31]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_test,y_pred_rf))
print(confusion_matrix(y_test,y_pred_rf))
print(classification_report(y_test,y_pred_rf))

0.9673384294649062
[[696  39]
 [  8 696]]
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       735
           1       0.95      0.99      0.97       704

    accuracy                           0.97      1439
   macro avg       0.97      0.97      0.97      1439
weighted avg       0.97      0.97      0.97      1439



In [32]:
import pickle 
pickle.dump(rf_classifier,open("D:/Model/rf_classifier.pkl","wb"))
pickle.dump(scaler,open("D:/Model/Scaler.pkl","wb"))
df.head()

,male,age,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [33]:
import numpy as np

def predict(rf_classifier, scaler, male, age, currentSmoker, cigsPerDay,
            BPMeds, prevalentStroke, prevalentHyp, diabetes,
            totChol, sysBP, diaBP, BMI, heartRate, glucose):

    male_encoded = 1 if male.lower() == 'male' else 0
    currentSmoker_encoded = 1 if currentSmoker.lower() == 'yes' else 0
    BPMeds_encoded = 1 if BPMeds.lower() == 'yes' else 0
    prevalentStroke_encoded = 1 if prevalentStroke.lower() == 'yes' else 0
    prevalentHyp_encoded = 1 if prevalentHyp.lower() == 'yes' else 0
    diabetes_encoded = 1 if diabetes.lower() == 'yes' else 0

    features = np.array([[male_encoded, age, currentSmoker_encoded, cigsPerDay,
                          BPMeds_encoded, prevalentStroke_encoded,
                          prevalentHyp_encoded, diabetes_encoded,
                          totChol, sysBP, diaBP, BMI, heartRate, glucose]])

    scaled_features = scaler.transform(features)

    result = rf_classifier.predict(scaled_features)

    return result[0]

In [35]:
male='female'
age=56.00
currentSmoker='yes'
cigsPerDay=3.00
BPMeds='no'
prevalentStroke='no'
prevalentHyp='yes'
diabetes='no'
totChol=285.00
sysBP=145.00
diaBP=100.00
BMI=30.14
heartRate=80.00
glucose=86.00
result=predict(rf_classifier, scaler, male, age, currentSmoker, cigsPerDay,
            BPMeds, prevalentStroke, prevalentHyp, diabetes,
            totChol, sysBP, diaBP, BMI, heartRate, glucose)


if result==1:
    print("This Patient has Heart Disease")
else:
     print("This Patient has not Heart Disease")

This Patient has not Heart Disease


C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
